# Labor 3 Vertiefung: Huffman und Quellen mit Gedächtnis

**Name:**  
**Datum:**  

Dieses Notebook ist für Gruppen gedacht, die mit dem Pflichtteil früh fertig sind.

Sie bauen hier einen einfachen Huffman-Encoder selbst und untersuchen danach denselben Text als Quelle mit Gedächtnis 2. Ordnung.

Wichtig: Es geht nicht darum, `gzip` nachzubauen. Der eigene Huffman-Encoder soll die Verbindung zwischen Wahrscheinlichkeiten, Codewortlängen und Entropie sichtbar machen.


## 1. Setup und Textdaten

Das Notebook nutzt dieselbe Datei wie das Pflichtnotebook: `./data/big.txt`.

Gearbeitet wird auf Byteebene. Ein Symbol ist also ein Bytewert zwischen `0` und `255`.


In [ ]:
import gzip
import heapq
import itertools
import math
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt

with open("./data/big.txt", "r", encoding="utf-8") as f:
    text = f.read()

text_data = text.encode("utf-8")
print(f"Zeichen im Text: {len(text):,}")
print(f"Bytes im Text:   {len(text_data):,}")
print(f"Verschiedene Bytewerte: {len(set(text_data))}")


## 2. Hilfsfunktionen

Diese Funktionen werden später für Entropie und Vergleiche benötigt.

TODO: Ergänzen Sie die Entropieformel auf Basis von Häufigkeiten.


In [ ]:
def entropie_aus_counts(counts):
    total = sum(counts.values())
    H = 0.0
    for count in counts.values():
        if count > 0:
            p = count / total
            # TODO: Ergänzen Sie den Beitrag -p*log2(p).
            H += ...
    return H

def gzip_size(data: bytes):
    return len(gzip.compress(data))

byte_counts = Counter(text_data)
H0 = entropie_aus_counts(byte_counts)
print(f"Entropie ohne Gedächtnis: {H0:.3f} bit/Byte")
print(f"Unkomprimiert: {8.0:.3f} bit/Byte")


# Teil A: Huffman-Encoder selbst bauen

Ein Huffman-Code entsteht durch wiederholtes Zusammenfassen der zwei seltensten Knoten.

In diesem Notebook werden Bäume sehr einfach dargestellt:

- Blatt: `("leaf", symbol)`
- innerer Knoten: `("node", left, right)`

TODO: Ergänzen Sie den Algorithmus so, dass aus Häufigkeiten ein Huffman-Baum entsteht.


In [ ]:
def leaf(symbol):
    return ("leaf", symbol)

def node(left, right):
    return ("node", left, right)

def is_leaf(tree):
    return tree[0] == "leaf"

def build_huffman_tree(counts):
    tie = itertools.count()
    heap = []
    
    for symbol, weight in counts.items():
        heapq.heappush(heap, (weight, next(tie), leaf(symbol)))
    
    if len(heap) == 1:
        return heap[0][2]
    
    while len(heap) > 1:
        # TODO: Entfernen Sie die zwei Knoten mit kleinster Häufigkeit.
        weight_left, _, left = ...
        weight_right, _, right = ...
        
        # TODO: Bauen Sie daraus einen Elternknoten und legen Sie ihn mit Summengewicht zurück in den Heap.
        parent = ...
        parent_weight = ...
        heapq.heappush(heap, (parent_weight, next(tie), parent))
    
    return heap[0][2]

huffman_tree = build_huffman_tree(byte_counts)
huffman_tree[0]


## 3. Codewörter aus dem Baum lesen

Der Weg nach links erhält hier eine `0`, der Weg nach rechts eine `1`.

TODO: Ergänzen Sie die rekursive Funktion.


In [ ]:
def build_code_table(tree, prefix="", codes=None):
    if codes is None:
        codes = {}
    
    if is_leaf(tree):
        symbol = tree[1]
        codes[symbol] = prefix or "0"
        return codes
    
    _, left, right = tree
    # TODO: Links mit angehängter 0, rechts mit angehängter 1 weitergehen.
    ...
    ...
    return codes

codes = build_code_table(huffman_tree)
print(f"Codewörter: {len(codes)}")
print("Kürzeste Länge:", min(len(c) for c in codes.values()))
print("Längste Länge:", max(len(c) for c in codes.values()))


In [ ]:
def byte_label(symbol):
    if 32 <= symbol <= 126:
        return repr(chr(symbol))
    return f"0x{symbol:02x}"

haeufigste = byte_counts.most_common(12)
for symbol, count in haeufigste:
    print(f"{byte_label(symbol):>6}  count={count:>8}  code={codes[symbol]}")


## 4. Präfixfreiheit prüfen

Ein Huffman-Code muss präfixfrei sein.

TODO: Ergänzen Sie die Prüfung. Kein Codewort darf Anfang eines anderen Codeworts sein.


In [ ]:
def ist_praefixfrei(code_table):
    codewords = list(code_table.values())
    for a in codewords:
        for b in codewords:
            # TODO: Wenn a und b verschieden sind und b mit a beginnt, ist der Code nicht präfixfrei.
            if ...:
                return False
    return True

print("präfixfrei:", ist_praefixfrei(codes))


## 5. Encoding und Decoding

Für den Test wird nur ein Ausschnitt des Textes wirklich als Bitstring codiert. Für den gesamten Text wird später die Länge über Häufigkeiten berechnet.

TODO: Ergänzen Sie Encoder und Decoder.


In [ ]:
def huffman_encode(data: bytes, code_table):
    # TODO: Ersetzen Sie jedes Symbol durch sein Codewort und fügen Sie die Strings zusammen.
    return ...

def huffman_decode(bitstring, tree):
    output = []
    current = tree
    for bit in bitstring:
        # TODO: Bei 0 nach links, bei 1 nach rechts gehen.
        current = ...
        if is_leaf(current):
            output.append(current[1])
            current = tree
    return bytes(output)

sample = text_data[:20_000]
encoded_sample = huffman_encode(sample, codes)
decoded_sample = huffman_decode(encoded_sample, huffman_tree)

print("Bits im Sample:", len(encoded_sample))
print("Decoding korrekt:", decoded_sample == sample)


## 6. Mittlere Codewortlänge und Größenvergleich

Die mittlere Codewortlänge ergibt sich aus den Symbolwahrscheinlichkeiten und den Codewortlängen.

TODO: Ergänzen Sie $L=\sum_i p_i l_i$.


In [ ]:
def mittlere_codewortlaenge(counts, code_table):
    total = sum(counts.values())
    L = 0.0
    for symbol, count in counts.items():
        p = count / total
        # TODO: Ergänzen Sie den gewichteten Längenbeitrag.
        L += ...
    return L

L = mittlere_codewortlaenge(byte_counts, codes)
huffman_bits_full = sum(count * len(codes[symbol]) for symbol, count in byte_counts.items())
huffman_bytes_without_table = math.ceil(huffman_bits_full / 8)

print(f"H0: {H0:.3f} bit/Byte")
print(f"L:  {L:.3f} bit/Byte")
print(f"Rohdaten:       {len(text_data):>10} Byte")
print(f"Huffman ohne Tabelle: {huffman_bytes_without_table:>10} Byte")
print(f"gzip:           {gzip_size(text_data):>10} Byte")
print(f"Abstand L-H0: {L-H0:.3f} bit/Byte")


**Kurze Deutung zu Teil A:**

1. Warum ist `L` nicht kleiner als `H0`?  
   ...

2. Warum ist die Huffman-Größe hier trotzdem nicht direkt mit einer echten Dateigröße gleichzusetzen?  
   ...

3. Warum kann `gzip` besser sein als dieser einfache Huffman-Code?  
   ...


# Teil B: Bedingte Entropie und Quelle mit Gedächtnis 2. Ordnung

Bisher wurde der Text wie eine gedächtnislose Quelle betrachtet: Jedes Byte wurde nur über seine Einzelhäufigkeit modelliert.

Jetzt wird gefragt:

- Wie unsicher ist das nächste Byte, wenn das vorherige Byte bekannt ist?
- Wie unsicher ist das nächste Byte, wenn die vorherigen zwei Bytes bekannt sind?

Für Ordnung 2 lautet die Größe:

$$
H(X_n \mid X_{n-2}, X_{n-1}).
$$


## 7. Folgezeichen pro Kontext zählen

Ein Kontext der Ordnung 2 besteht aus den zwei vorherigen Bytes.

Beispiel: Wenn im Text oft `th` vorkommt, ist das nächste Zeichen nicht gleichverteilt.

TODO: Ergänzen Sie die Zählung der Folgezeichen.


In [ ]:
def follower_counts(sequence: bytes, order: int):
    followers = defaultdict(Counter)
    for i in range(order, len(sequence)):
        # TODO: Kontext sind die order vorherigen Symbole.
        context = ...
        next_symbol = sequence[i]
        followers[context][next_symbol] += 1
    return followers

followers_order_1 = follower_counts(text_data, order=1)
followers_order_2 = follower_counts(text_data, order=2)

print("Kontexte Ordnung 1:", len(followers_order_1))
print("Kontexte Ordnung 2:", len(followers_order_2))


## 8. Bedingte Entropie berechnen

Die bedingte Entropie ist ein gewichteter Mittelwert über alle Kontexte:

$$
H(X_n \mid Kontext)=\sum_c P(c)H(X_n\mid c).
$$

TODO: Ergänzen Sie die Gewichtung der Kontextentropien.


In [ ]:
def conditional_entropy(sequence: bytes, order: int):
    followers = follower_counts(sequence, order)
    total_context_occurrences = sum(sum(counts.values()) for counts in followers.values())
    H_cond = 0.0
    
    for context, counts in followers.items():
        context_count = sum(counts.values())
        H_given_context = entropie_aus_counts(counts)
        # TODO: Gewicht des Kontextes berechnen und gewichteten Beitrag addieren.
        weight = ...
        H_cond += ...
    
    return H_cond

H1 = conditional_entropy(text_data, order=1)
H2 = conditional_entropy(text_data, order=2)

print(f"H(X_n)                         = {H0:.3f} bit/Byte")
print(f"H(X_n | X_(n-1))               = {H1:.3f} bit/Byte")
print(f"H(X_n | X_(n-2), X_(n-1))      = {H2:.3f} bit/Byte")
print(f"Reduktion von H0 zu H2:          {H0-H2:.3f} bit/Byte")


## 9. Häufige Kontexte ansehen

Nicht alle Kontexte sind gleich informativ. Häufige Kontexte zeigen oft gut, warum Sprache Gedächtnis besitzt.


In [ ]:
def context_label(context):
    return bytes(context).decode("utf-8", errors="replace")

def symbol_label(symbol):
    return bytes([symbol]).decode("utf-8", errors="replace")

haeufige_kontexte = sorted(
    followers_order_2.items(),
    key=lambda item: sum(item[1].values()),
    reverse=True
)[:10]

for context, counts in haeufige_kontexte:
    top_followers = counts.most_common(5)
    followers_text = ", ".join(f"{symbol_label(s)!r}:{c}" for s, c in top_followers)
    print(f"Kontext {context_label(context)!r:>8} -> {followers_text}")


**Kurze Deutung zu Teil B:**

1. Warum sollte im Idealfall gelten: $H(X_n \mid X_{n-2},X_{n-1}) \le H(X_n \mid X_{n-1}) \le H(X_n)$?  
   ...

2. Welche zwei Byte-Kontexte waren besonders häufig? Was sagt das über den Text aus?  
   ...

3. Warum erklärt eine Quelle mit Gedächtnis besser, warum natürliche Sprache komprimierbar ist?  
   ...


## 10. Ergebnis für schnelle Gruppen

Fassen Sie knapp zusammen:

- Der selbst gebaute Huffman-Code erreicht `L = ... bit/Byte`.
- Die Entropie ohne Gedächtnis beträgt `H0 = ... bit/Byte`.
- Die bedingte Entropie zweiter Ordnung beträgt `H2 = ... bit/Byte`.
- Der Unterschied zwischen Huffman und Quelle mit Gedächtnis ist ...
- Eine anspruchsvolle Klausurfrage dazu wäre ...
